Imports

In [99]:
%pip install -q matplotlib seaborn xgboost scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [100]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os

Create dataframe and initialize X and Y

In [101]:
path = "data/modeling_dataset.csv"

base_dir = os.path.dirname(os.path.abspath(os.getcwd()))

df = pd.read_csv(os.path.join(base_dir, path))

X = df.drop(columns=["fantasy_points"])

# bucket points into 4 ordered tiers: bench / ok / good / haul                                                                        
y = pd.cut(df["fantasy_points"].clip(lower=0),                                                                                        
            bins=[-1, 1, 4, 8, 100],                   
            labels=[0, 1, 2, 3]).astype(int)   

Print first 5 rows to see some data

In [102]:
df.head(5)

,player,team,opponent,home_away,fantasy_position,season,matchday,date,fantasy_points_prev,fantasy_points_last3_avg,fantasy_points_last5_avg,minutes_prev,minutes_last3_avg,minutes_last5_avg,goals_prev,goals_last3_avg,goals_last5_avg,assists_prev,assists_last3_avg,assists_last5_avg,shots_prev,shots_last3_avg,shots_last5_avg,shots_on_target_prev,shots_on_target_last3_avg,shots_on_target_last5_avg,xg_prev,xg_last3_avg,xg_last5_avg,npxg_prev,npxg_last3_avg,npxg_last5_avg,xag_prev,xag_last3_avg,xag_last5_avg,sca_prev,sca_last3_avg,sca_last5_avg,gca_prev,gca_last3_avg,...,ball_recoveries_last3_avg,ball_recoveries_last5_avg,yellow_cards_prev,yellow_cards_last3_avg,yellow_cards_last5_avg,red_cards_prev,red_cards_last3_avg,red_cards_last5_avg,fouls_prev,fouls_last3_avg,fouls_last5_avg,saves_prev,saves_last3_avg,saves_last5_avg,penalties_saved_prev,penalties_saved_last3_avg,penalties_saved_last5_avg,goals_conceded_prev,goals_conceded_last3_avg,goals_conceded_last5_avg,clean_sheet_prev,clean_sheet_last3_avg,clean_sheet_last5_avg,own_goals_prev,own_goals_last3_avg,own_goals_last5_avg,penalties_won_prev,penalties_won_last3_avg,penalties_won_last5_avg,pens_made_prev,pens_made_last3_avg,pens_made_last5_avg,pens_att_prev,pens_att_last3_avg,pens_att_last5_avg,played_prev,played_60_prev,minutes_last3_sum,regular_recently,fantasy_points
0,Aaron Anselmino,Dortmund de,es Villarreal,home,DEF,2025-2026,10,2025-11-25,0.0,0.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,6.0,0,9
1,Aaron Anselmino,Dortmund de,no Bodø/Glimt,home,DEF,2025-2026,13,2025-12-10,9.0,4.5,4.5,90.0,48.0,48.0,0.0,0.0,0.0,1.0,0.5,0.5,1.0,0.5,0.5,1.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,96.0,0,1
2,Aaron Bouwman,nl Ajax,Villarreal es,away,DEF,2025-2026,14,2026-01-20,1.0,1.0,1.0,90.0,90.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,90.0,0,1
3,Aaron Mooy,Celtic sct,de RB Leipzig,home,MID,2022-2023,7,2022-10-11,1.0,1.0,1.0,23.0,23.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,23.0,0,1
4,Aaron Mooy,Celtic sct,ua Shakhtar Donetsk,home,MID,2022-2023,9,2022-10-25,1.0,1.0,1.0,25.0,24.0,24.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,1.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,48.0,0,1


Display X and y

In [103]:
X.head(5)

,player,team,opponent,home_away,fantasy_position,season,matchday,date,fantasy_points_prev,fantasy_points_last3_avg,fantasy_points_last5_avg,minutes_prev,minutes_last3_avg,minutes_last5_avg,goals_prev,goals_last3_avg,goals_last5_avg,assists_prev,assists_last3_avg,assists_last5_avg,shots_prev,shots_last3_avg,shots_last5_avg,shots_on_target_prev,shots_on_target_last3_avg,shots_on_target_last5_avg,xg_prev,xg_last3_avg,xg_last5_avg,npxg_prev,npxg_last3_avg,npxg_last5_avg,xag_prev,xag_last3_avg,xag_last5_avg,sca_prev,sca_last3_avg,sca_last5_avg,gca_prev,gca_last3_avg,...,ball_recoveries_prev,ball_recoveries_last3_avg,ball_recoveries_last5_avg,yellow_cards_prev,yellow_cards_last3_avg,yellow_cards_last5_avg,red_cards_prev,red_cards_last3_avg,red_cards_last5_avg,fouls_prev,fouls_last3_avg,fouls_last5_avg,saves_prev,saves_last3_avg,saves_last5_avg,penalties_saved_prev,penalties_saved_last3_avg,penalties_saved_last5_avg,goals_conceded_prev,goals_conceded_last3_avg,goals_conceded_last5_avg,clean_sheet_prev,clean_sheet_last3_avg,clean_sheet_last5_avg,own_goals_prev,own_goals_last3_avg,own_goals_last5_avg,penalties_won_prev,penalties_won_last3_avg,penalties_won_last5_avg,pens_made_prev,pens_made_last3_avg,pens_made_last5_avg,pens_att_prev,pens_att_last3_avg,pens_att_last5_avg,played_prev,played_60_prev,minutes_last3_sum,regular_recently
0,Aaron Anselmino,Dortmund de,es Villarreal,home,DEF,2025-2026,10,2025-11-25,0.0,0.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,6.0,0
1,Aaron Anselmino,Dortmund de,no Bodø/Glimt,home,DEF,2025-2026,13,2025-12-10,9.0,4.5,4.5,90.0,48.0,48.0,0.0,0.0,0.0,1.0,0.5,0.5,1.0,0.5,0.5,1.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,96.0,0
2,Aaron Bouwman,nl Ajax,Villarreal es,away,DEF,2025-2026,14,2026-01-20,1.0,1.0,1.0,90.0,90.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,90.0,0
3,Aaron Mooy,Celtic sct,de RB Leipzig,home,MID,2022-2023,7,2022-10-11,1.0,1.0,1.0,23.0,23.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,23.0,0
4,Aaron Mooy,Celtic sct,ua Shakhtar Donetsk,home,MID,2022-2023,9,2022-10-25,1.0,1.0,1.0,25.0,24.0,24.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,1.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,48.0,0


In [104]:
y.head(5)

0    3
1    0
2    0
3    0
4    0
Name: fantasy_points, dtype: int32

From the X matrix, we can see that there is a lot of significant and insignificant data that are in a nonumeric format.

In [105]:
# Preprocessing for converting text columns into numeric 
# drop name, date, season, team, opponent (text columns)
X_clean = X.drop(columns=["player", "date", "season", "team", "opponent"])

# turning home and away into binary values
X_clean["home_away"] = (X_clean["home_away"] == "home").astype(int)

# every position column, marked 1 if it matches that row and 0 otherwise  
X_clean = pd.get_dummies(X_clean, columns=["fantasy_position"], drop_first=True)                                                      
X_clean = X_clean.astype(float) 

print("Shape: ", X_clean.shape)
print("non-numeric columns left", X_clean.select_dtypes(exclude="number").columns.tolist())
X_clean.head()

Shape:  (16549, 87)
non-numeric columns left []


,home_away,matchday,fantasy_points_prev,fantasy_points_last3_avg,fantasy_points_last5_avg,minutes_prev,minutes_last3_avg,minutes_last5_avg,goals_prev,goals_last3_avg,goals_last5_avg,assists_prev,assists_last3_avg,assists_last5_avg,shots_prev,shots_last3_avg,shots_last5_avg,shots_on_target_prev,shots_on_target_last3_avg,shots_on_target_last5_avg,xg_prev,xg_last3_avg,xg_last5_avg,npxg_prev,npxg_last3_avg,npxg_last5_avg,xag_prev,xag_last3_avg,xag_last5_avg,sca_prev,sca_last3_avg,sca_last5_avg,gca_prev,gca_last3_avg,gca_last5_avg,tackles_prev,tackles_last3_avg,tackles_last5_avg,interceptions_prev,interceptions_last3_avg,...,yellow_cards_prev,yellow_cards_last3_avg,yellow_cards_last5_avg,red_cards_prev,red_cards_last3_avg,red_cards_last5_avg,fouls_prev,fouls_last3_avg,fouls_last5_avg,saves_prev,saves_last3_avg,saves_last5_avg,penalties_saved_prev,penalties_saved_last3_avg,penalties_saved_last5_avg,goals_conceded_prev,goals_conceded_last3_avg,goals_conceded_last5_avg,clean_sheet_prev,clean_sheet_last3_avg,clean_sheet_last5_avg,own_goals_prev,own_goals_last3_avg,own_goals_last5_avg,penalties_won_prev,penalties_won_last3_avg,penalties_won_last5_avg,pens_made_prev,pens_made_last3_avg,pens_made_last5_avg,pens_att_prev,pens_att_last3_avg,pens_att_last5_avg,played_prev,played_60_prev,minutes_last3_sum,regular_recently,fantasy_position_FWD,fantasy_position_GK,fantasy_position_MID
0,1.0,10.0,0.0,0.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.0,0.0,0.0,0.0,0.0
1,1.0,13.0,9.0,4.5,4.5,90.0,48.0,48.0,0.0,0.0,0.0,1.0,0.5,0.5,1.0,0.5,0.5,1.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,96.0,0.0,0.0,0.0,0.0
2,0.0,14.0,1.0,1.0,1.0,90.0,90.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,90.0,0.0,0.0,0.0,0.0
3,1.0,7.0,1.0,1.0,1.0,23.0,23.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,23.0,0.0,0.0,0.0,1.0
4,1.0,9.0,1.0,1.0,1.0,25.0,24.0,24.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,1.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,48.0,0.0,0.0,0.0,1.0


In [106]:
# train test split
order = df.sort_values(by="date").index
X_sorted = X_clean.loc[order].reset_index(drop=True)
y_sorted = y.loc[order].reset_index(drop=True)

# train on first 80% of games and last 20% for testing
split_indx = int(len(X_sorted) * 0.8)
X_train, X_test = X_sorted.iloc[:split_indx], X_sorted.iloc[split_indx:]
y_train, y_test = y_sorted.iloc[:split_indx], y_sorted.iloc[split_indx:]

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)
 

Training set shape: (13239, 87)
Testing set shape: (3310, 87)


Train Multiple Multiple linear Regression Model

In [107]:
model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

Evaluate the model

In [108]:
print("MAE:", mean_absolute_error(y_test, y_pred))

print("MSE:", mean_squared_error(y_test, y_pred))

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

print("R2:", r2_score(y_test, y_pred))

MAE: 0.6594749372712517
MSE: 0.6491882501338764
RMSE: 0.8057221916602995
R2: 0.15603432223659486
